# CMIP6 decadal daily concat and subset with batched outputs

**Date:** 2026-08-21

This example demonstrates that daily **CMIP6 decadal** data can be concatenated by realization and then subset through the CDS Rook service. The request selects the autumn months (September–November) of 1962–1964 over Europe from ten EC-Earth3 hindcast realizations.

Large collections are processed in batches. Consequently, one workflow response can contain several NetCDF output files. The cells below show how to discover, download, and inspect every file instead of assuming that a request always returns one file.

## Configure Rooki

In [1]:
import os
from pathlib import Path
from time import perf_counter
from urllib.parse import unquote, urlparse

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

import pandas as pd
import xarray as xr

from rooki import operators as ops

## Select the daily decadal collection

Each dataset identifier represents one realization initialized in 1961. The workflow first concatenates them along the `realization` dimension and then applies the spatiotemporal subset. This is the relevant decadal workflow and exercises the server-side batching used to keep the operation within its memory limits.

Rook's `area` order is `west,south,east,north`; the bounding box below covers Europe.

In [2]:
dataset_ids = [
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r9i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216",
]

time_range = "1962/1964"
autumn_months = "month:09,10,11"
europe = "-10,30,35,70"

## Submit the concat + subset workflow

The result is a single workflow response whose Metalink document describes all batch output files. The elapsed time measures the complete server-side orchestration as observed by the client.

In [3]:
psl = ops.Input("psl", dataset_ids)
concatenated = ops.Concat(psl, dims="realization")
workflow = ops.Subset(
    concatenated,
    time=time_range,
    time_components=autumn_months,
    area=europe,
)

started_at = perf_counter()
response = workflow.orchestrate()
elapsed_seconds = perf_counter() - started_at

print(f"Orchestration time: {elapsed_seconds:.1f} seconds ({elapsed_seconds / 60:.1f} minutes)")
print(response.status)
assert response.ok, response.status

Orchestration time: 30.9 seconds (0.5 minutes)
ProcessSucceeded


## Inspect the batched output manifest

`response.num_files` is the number of NetCDF results, not the number of workflow responses. The table makes every batched output part visible and preserves the exact server-generated filenames.

In [4]:
urls = response.download_urls()

output_manifest = pd.DataFrame(
    {
        "part": range(1, len(urls) + 1),
        "filename": [unquote(Path(urlparse(url).path).name) for url in urls],
        "url": urls,
    }
)

print(f"{response.num_files} output file(s), {response.size_in_mb:.2f} MiB in total")
output_manifest

3 output file(s), 28.45 MiB in total


,part,filename,url
0,1,psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_...,http://rook7.cloud.dkrz.de:80/outputs/rook/24f...
1,2,psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_...,http://rook7.cloud.dkrz.de:80/outputs/rook/24f...
2,3,psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_...,http://rook7.cloud.dkrz.de:80/outputs/rook/24f...


## Download and inspect every output file

Do not select only `response.datasets()[0]`: with batching that would silently ignore the remaining results. Here each downloaded file is opened independently and summarized. The checks verify that every output part contains only the requested dates and geographic region.

In [5]:
downloaded_files = [Path(filename) for filename in response.download()]
assert len(downloaded_files) == response.num_files

summaries = []
for part, filename in enumerate(downloaded_files, start=1):
    with xr.open_dataset(filename, decode_timedelta=True) as ds:
        assert str(ds.time.min().dt.strftime("%Y-%m-%d").item()) >= "1962-01-01"
        assert str(ds.time.max().dt.strftime("%Y-%m-%d").item()) <= "1964-12-31"
        assert set(ds.time.dt.month.values.tolist()) <= {9, 10, 11}
        assert float(ds.lat.min()) >= 30
        assert float(ds.lat.max()) <= 70

        summaries.append(
            {
                "part": part,
                "filename": filename.name,
                "size_mib": filename.stat().st_size / 1024**2,
                "dimensions": dict(ds.sizes),
                "time_start": str(ds.time.min().dt.strftime("%Y-%m-%d").item()),
                "time_end": str(ds.time.max().dt.strftime("%Y-%m-%d").item()),
                "realizations": ds["realization"].values.tolist() if "realization" in ds else None,
                "variables": ", ".join(ds.data_vars),
            }
        )

pd.DataFrame(summaries)

,part,filename,size_mib,dimensions,time_start,time_end,realizations,variables
0,1,psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_...,9.472092,"{'realization': 10, 'lat': 57, 'bnds': 2, 'lon...",1962-09-01,1962-11-30,"[10, 1, 2, 3, 4, 5, 6, 7, 8, 9]","lat_bnds, lon_bnds, psl"
1,2,psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_...,9.493271,"{'realization': 10, 'lat': 57, 'bnds': 2, 'lon...",1963-09-01,1963-11-30,"[10, 1, 2, 3, 4, 5, 6, 7, 8, 9]","lat_bnds, lon_bnds, psl"
2,3,psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_...,9.480526,"{'realization': 10, 'lat': 57, 'bnds': 2, 'lon...",1964-09-01,1964-11-30,"[10, 1, 2, 3, 4, 5, 6, 7, 8, 9]","lat_bnds, lon_bnds, psl"


## Show the Woodpecker decadal fixes

Rook applies the CMIP6-decadal Woodpecker recipe automatically. The first batched file is representative: the following tables show the corrected realization axis, forecast coordinates, start date, calendar, and model descriptions.

In [6]:
from IPython.display import display

representative = xr.open_dataset(downloaded_files[0], decode_timedelta=True)
coordinate_metadata = pd.DataFrame(
    [
        {
            "coordinate": name,
            "dtype": str(representative[name].dtype),
            "long_name": representative[name].attrs.get("long_name"),
            "standard_name": representative[name].attrs.get("standard_name"),
        }
        for name in ("time", "realization", "reftime", "leadtime")
    ]
)

print("Realizations:", sorted(representative.realization.values.tolist()))
display(coordinate_metadata)

Realizations: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


,coordinate,dtype,long_name,standard_name
0,time,datetime64[ns],valid_time,time
1,realization,int32,None,realization
2,reftime,datetime64[ns],Start date of the forecast,forecast_reference_time
3,leadtime,timedelta64[ns],Time elapsed since the start of the forecast,forecast_period


In [7]:
global_attributes = pd.Series(
    {
        "calendar": representative.time.dt.calendar,
        "startdate": representative.attrs.get("startdate"),
        "sub_experiment_id": representative.attrs.get("sub_experiment_id"),
        "realization_index": representative.attrs.get("realization_index"),
        "forcing_description": representative.attrs.get("forcing_description"),
        "physics_description": representative.attrs.get("physics_description"),
        "initialization_description": representative.attrs.get("initialization_description"),
    },
    name="value",
).to_frame()

display(global_attributes)
representative.close()

,value
calendar,proleptic_gregorian
startdate,s196111
sub_experiment_id,s196111
realization_index,10
forcing_description,"f1, CMIP6 historical forcings"
physics_description,"physics from the standard model configuration,..."
initialization_description,Atmosphere initialization based on full-fields...
